Read Engineered Features
Perform train test split
Remove relevant columns to avoid feature leak


In [15]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from datetime import timedelta
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("agg_full_old.csv", parse_dates=["order_date"])
df = df.sort_values(["coating_id", "order_date"])

import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from datetime import timedelta

# Sicherstellen, dass Kategorientypen erkennbar sind
cat_cols = df.select_dtypes(include="object").columns.tolist()

results = []

coating_ids = df['coating_id'].unique()

for coating_id in coating_ids:
    df_coating = df[df['coating_id'] == coating_id].copy()

    # Zielspalte erzeugen
    df_coating["target_fill_rate_next_day"] = df_coating["total_fill_rate"].shift(-1)
    df_coating = df_coating.dropna(subset=["target_fill_rate_next_day"])

    # Train/Test split
    last_train_date = df_coating["order_date"].max() - timedelta(days=30)
    train = df_coating[df_coating["order_date"] <= last_train_date]
    test = df_coating[df_coating["order_date"] > last_train_date]

    if len(train) < 10 or len(test) < 5:
        continue

    drop_cols = ["order_date", "total_fill_rate", "target_fill_rate_next_day"]
    target = "target_fill_rate_next_day"
    features = [col for col in df.columns if col not in drop_cols]

    # Kategorie-Typen setzen
    for col in cat_cols:
        train[col] = train[col].astype("category")
        test[col] = test[col].astype("category")

    X_train = train[features]
    y_train = train[target]
    X_test = test[features]
    y_test = test[target]

    lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=cat_cols)
    lgb_test = lgb.Dataset(X_test, y_test, reference=lgb_train)

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt'
    }

    model = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_test],
    )

    y_pred = model.predict(X_test, num_iteration=model.best_iteration)

    rmse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "coating_id": coating_id,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "n_test_samples": len(y_test)
    })

# Ergebnisse anzeigen
results_df = pd.DataFrame(results).sort_values("rmse")
print(results_df)


    coating_id          rmse         mae          r2  n_test_samples
11          13      0.114309    0.204929 -676.995506              21
14          20      0.287883    0.403183    0.000000              21
21          28      0.315178    0.410851   -4.805751              21
32          80      0.769519    0.740717   -0.232531              21
18          24      1.802040    0.937313   -1.118748              21
15          21      4.579924    1.319203   -9.943926              21
9           10      4.865816    1.067805   -0.119451              21
26          46      5.408452    1.700698    0.344522              21
4            5      6.563398    2.159793   -0.518126              21
19          25     13.198783    2.430428  -55.007025              21
29          59     15.153169    2.776088  -10.469866              21
31          69     16.817448    2.480919   -0.142456              21
13          16     28.007182    4.183665   -2.901353              21
1            2     30.098699    4.

In [16]:
print(results_df.mae.mean())

27.070470335662645
